# R1.4 — сохранение и возобновление
Проверка механизма, не полная научная карта. Первые примеры — заданные синтетические наблюдения; последний использует короткий поток действующего ядра.

In [ ]:
from pathlib import Path
import os, sys, json, tempfile
root=Path.cwd()
while root != root.parent and not (root/'efc/model.py').exists():root=root.parent
assert (root/'efc/model.py').exists()
os.chdir(root);sys.path.insert(0,str(root));sys.path.insert(0,str(root/'tests'))
from dataclasses import replace
from efc.model import Config
from efc.grid import GridRunner, CachedSampler, read_json
from efc.pilot import KernelSampler
from test_grid import FixtureSampler
out=root/'results/r14/current/grid';out.mkdir(parents=True,exist_ok=True)
print('Проверяем сохранение, не научную допустимость.')

Две искусственные клетки проверяют правило выбора и запрет повторных опытов. После неудачного подтверждения следующий кандидат не подбирается.

In [ ]:
c=Config(n_cloud=40,n_fog=30,buffer=2,arrival_rate_s=.1*50/.055)
FixtureSampler.calls=[];FixtureSampler.fail_seed=None;FixtureSampler.bad_confirmation=True
with tempfile.TemporaryDirectory() as d:
    configs=[c,replace(c,n_fog=40)]
    first=GridRunner(d,configurations=configs,factory=FixtureSampler).run(confirm=True)
    n=len(FixtureSampler.calls)
    second=GridRunner(d,configurations=configs,factory=FixtureSampler).run(confirm=True)
    assert len(FixtureSampler.calls)==n and second['new_replications']==0
    registry=read_json(Path(d)/'candidates.json')
    check=read_json(Path(d)/'cells/r0.10_F30_B002/confirmation.json')
    assert check['status']=='UNACCEPTABLE'
    assert not (Path(d)/'cells/r0.10_F40_B002/confirmation.json').exists()
    report={'scope':'synthetic controller oracle, NOT scientific map','first':first,
            'resumed':second,'registry':registry,'confirmation_status':check['status']}
(out/'resume-oracle.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'
')
print('Повторных запусков: 0. Отрицательное подтверждение сохранено; следующего кандидата нет.')

Короткий опыт на едином ядре сохраняется на диск и читается новым объектом без повторного моделирования. Окна 20/40/10 служат только технической проверке; они не заменяют утверждённые окна серии.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    actual=CachedSampler(KernelSampler(c),d)
    a=actual(100,20,40,10)
    restored=CachedSampler(KernelSampler(c),d)
    b=restored(100,20,40,10)
    assert a==b and actual.new_runs==1 and restored.new_runs==0 and restored.reused_runs==1
    evidence={'scope':'short kernel cache regression only','arrivals':a.full.arrivals,
              'dropped':a.full.dropped,'unfinished':a.full.unfinished,
              'provenance':a.provenance,'new_runs':actual.new_runs,
              'new_runs_after_restart':restored.new_runs,'reused_runs':restored.reused_runs}
(out/'kernel-cache.json').write_text(json.dumps(evidence,ensure_ascii=False,indent=2)+'
')
print(json.dumps(evidence,ensure_ascii=False,indent=2))
print('OK: механизм сохранения проверен. Карта 270×10 здесь не рассчитывалась.')